# 附录B：符号、公式与算法选择

In [1]:
from pathlib import Path
import sys
root = Path.cwd().resolve()
while not (root / 'lsq').is_dir():
    if root == root.parent:
        raise RuntimeError('Run inside the repository')
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from lsq import (finite_difference, whiten, gauss_newton,
                 levenberg_marquardt, dogleg, recursive_update)
np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update({'figure.figsize': (6.4, 3.3), 'font.size': 10,
                     'axes.grid': True, 'savefig.dpi': 130})

## 符号速查

| 符号 | 含义 | 维度 |
|---|---|---|
| $x,h$ | 参数、参数更新 | $n$ |
| $y,r$ | 观测、模型减观测的残差 | $m$ |
| $A,J$ | 线性设计矩阵、非线性残差雅可比 | $m\times n$ |
| $F,g,H$ | 半平方和、梯度、精确Hessian | 标量、$n$、$n\times n$ |
| $C$ | 观测噪声协方差 | $m\times m$ |
| $\mu,\Delta$ | LM阻尼、信赖域半径 | 标量 |
| $\lambda,D$ | 正则强度、惩罚算子 | 标量、按问题指定 |

RLS章的$\lambda$表示遗忘因子，语境与第06章的正则强度不同。符号相同不代表量的含义相同。

## 选择方法前先选择目标

| 情况 | 起点 | 仍需检查 |
|---|---|---|
| 线性、规模适中 | QR或SVD/lstsq | 秩、条件数、参数单位 |
| 已知观测协方差 | 白化后QR/SVD | 协方差来源、正定性 |
| 弱可辨识、明确先验 | 增广正则最小二乘 | 惩罚尺度、验证集 |
| 光滑非线性、初值合理 | GN加回溯 | 雅可比、模型适用范围 |
| 无界光滑非线性 | LM | 缩放、阻尼历史、局部性 |
| 步长需明确限制 | 信赖域/狗腿 | 预测下降与真实下降 |
| 有上下界或稳健损失 | SciPy trf | 可行初值、损失尺度 |
| 线性数据连续到来 | RLS | 参数变化、遗忘与数值漂移 |
| 量化、平台、含测量噪声 | 测量设计加受限局部校正 | 有效扰动、重复测量、可达性 |

## 六项交付检查

明确模型和单位；核对残差与雅可比；查看秩和尺度；记录停止原因；检查未参与拟合的数据；把概率近似和工程容差分别说明。依次做这些检查通常比不断更换求解器更有效。

## 公式速查

线性驻点：$A^T(Ax-y)=0$。非线性梯度：$g=J^Tr$。GN：$\min_h\|Jh+r\|^2$。LM：$\min_h\|Jh+r\|^2+\mu\|h\|^2$。预测下降：$-g^Th-\tfrac12\|Jh\|^2$。参数缩放$x=Sz$后：$J_z=J_xS$。

## 一段可以运行的初步诊断

In [2]:
def diagnose_matrix(A):
    s=np.linalg.svd(A,compute_uv=False)
    rank=np.linalg.matrix_rank(A)
    full_column_rank=rank==A.shape[1]
    condition=(s[0]/s[-1]) if full_column_rank else np.inf
    return {'shape':A.shape,'rank':rank,'full_column_rank':full_column_rank,
            'condition':condition}
print(diagnose_matrix(np.column_stack([np.ones(5),np.arange(5.)])))
print(diagnose_matrix(np.ones((5,2))))
assert not diagnose_matrix(np.ones((5,2)))['full_column_rank']

{'shape': (5, 2), 'rank': np.int64(2), 'full_column_rank': np.True_, 'condition': np.float64(4.738720018687268)}
{'shape': (5, 2), 'rank': np.int64(1), 'full_column_rank': np.False_, 'condition': inf}


练习：若矩阵诊断很好，是否说明残差没有模型偏差？解答：不说明；矩阵诊断只描述当前线性信息结构，残差诊断检验的是数据与模型之间的关系。